# 14.6 - LLMOps Capstone: Ship a Production Pipeline

**Module 14 - LLMOps** | Netsetos GenAI Engineering

This notebook builds one operated feature end to end: a QuickBite food-delivery review classifier wired behind every LLMOps practice from the module - deterministic A/B routing, a versioned prompt registry with a cacheable prefix, a full trace record on every call, a golden-set eval graded by an LLM judge, a three-check CI gate, and a cost projection. Each cell is a self-contained, keyless reference for one subsystem of the request path or the CI path.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

The capstone code is intentionally dependency-light so every cell runs keyless. Only the whole-game endpoint would need the `anthropic` SDK in production; here every subsystem is simulated with the Python standard library so you can run and read the wiring without a key or a network call.

In [ ]:
# Install dependencies (if running in Colab or fresh environment)
# Uncomment the next line if needed:
# !pip install anthropic -q

# Reproducibility - the lesson uses seeded random values throughout

**Explanation**

This is environment prep, not a model call. The `pip install` line is commented out because none of the runnable cells actually import `anthropic` - the standard-library `hashlib` (used by the router) is the only real dependency, and it ships with Python.

**How the code works - step by step**
- **Commented `pip install anthropic`** - uncomment only if you extend the notebook to make a real API call; the runnable cells below do not need it.
- **Reproducibility note** - flags that the lesson uses seeded, deterministic values throughout, so every run prints the same numbers.

**In one line:** no packages to install - the cells are self-contained and keyless.

**What you'll see:** (no output - environment prep)

## 1 - The whole game: one request, six stages

Before building any subsystem, see the entire capstone at once. A single `POST /classify` request is not one API call - it is a small pipeline of six wired decisions returning a fixed JSON contract. This cell runs that pipeline for two different requests so you can watch the routing change while the path stays identical.

In [ ]:
# THE WHOLE GAME: one POST /classify request threads through SIX wired stages and returns the JSON contract.
# This is the entire capstone in miniature - route -> registry -> model pick -> render+cache -> instrumented call -> parse.
import hashlib
PROMPTS = {"v1.0.0": "baseline", "v1.1.0": "few-shot"}          # the prompt registry (from Lesson 14.2)
def bucket(user_id, candidate_pct=10):                          # (1) deterministic A/B pick by user_id
    return "v1.1.0" if int(hashlib.md5(user_id.encode()).hexdigest(), 16) % 100 < candidate_pct else "v1.0.0"
def pick_model(review):                                         # (3) cost routing by length (from Lesson 14.5)
    return "claude-haiku-4-5" if len(review) < 100 else "claude-sonnet-4-6"
def classify(review, user_id):
    version = bucket(user_id)                                   # (1) which prompt version
    prompt = PROMPTS[version]                                   # (2) load it from the registry
    model = pick_model(review)                                  # (3) which model
    # (4) render template + cache_control on the prefix, (5) instrumented Anthropic call, (6) parse - simulated here:
    category, confidence = ("refund", 0.94) if "refund" in review.lower() else ("delivery", 0.88)
    return {"category": category, "confidence": confidence, "prompt_version": version, "model": model}
print("short refund review ->", classify("Refund never arrived, awful.", "user_0001"))                    # champion + Haiku
print("long delivery review->", classify("The driver left it at the wrong building entirely and it took another full hour with support to sort out the mix-up.", "user_0017"))  # candidate + Sonnet
print()
print("Six stages, every request: A/B pick -> registry -> model route -> render+cache -> instrumented call -> parse.")
print("The whole capstone is wiring these six into one endpoint that LOGS everything and GATES every prompt change.")

# Output:
# short refund review -> {'category': 'refund', 'confidence': 0.94, 'prompt_version': 'v1.0.0', 'model': 'claude-haiku-4-5'}
# long delivery review-> {'category': 'delivery', 'confidence': 0.88, 'prompt_version': 'v1.1.0', 'model': 'claude-sonnet-4-6'}
#
# Six stages, every request: A/B pick -> registry -> model route -> render+cache -> instrumented call -> parse.
# The whole capstone is wiring these six into one endpoint that LOGS everything and GATES every prompt change.

**Explanation**

This is the capstone in miniature - a single `classify()` function that threads six stages (A/B pick, registry load, model route, render+cache, instrumented call, parse) and returns `{category, confidence, prompt_version, model}`. Stages 4-6 are simulated inline here; the point is the shape of the path, not a real model response. Read it as: every request takes the same six-station route, and only the routing decisions differ per request.

**How the code works - step by step**
- **`PROMPTS`** - the two-entry prompt registry standing in for the versioned YAML files (baseline `v1.0.0`, few-shot `v1.1.0`).
- **`bucket`** - stage (1), the deterministic A/B pick: hashes `user_id` with md5, takes it mod 100, and routes below `candidate_pct` (10) to the candidate version.
- **`pick_model`** - stage (3), cost routing: reviews under 100 chars go to Haiku, longer ones to Sonnet.
- **`classify`** - the glue: buckets the version, loads the prompt, picks the model, then simulates stages (4)-(6) - render+cache, instrumented call, parse - and returns the JSON contract.
- **Two `print` calls** - run a short refund review (`user_0001`) and a long delivery review (`user_0017`) through the same path.

**In one line:** route -> load -> pick model -> render+cache -> call -> parse, returning one JSON contract per request.

**What you'll see:** The short refund review returns `v1.0.0` + `claude-haiku-4-5`; the long delivery review returns `v1.1.0` + `claude-sonnet-4-6` - same path, different routing - followed by two lines summarizing the six stages and the two things the capstone wires: logging every request and gating every prompt change.

## 2 - The router: two deterministic decisions

Zoom into the first subsystem. Per request the router makes exactly two calls - which prompt VERSION (the A/B decision) and which MODEL (the cost decision) - and the whole point is that both are deterministic. This cell runs the version decision over 1000 users and proves a single user never flips.

In [ ]:
# THE ROUTER: two deterministic decisions per request - which prompt VERSION (A/B) and which MODEL (cost routing).
import hashlib
def bucket(user_id, candidate_pct=10):
    return "candidate v1.1.0" if int(hashlib.md5(user_id.encode()).hexdigest(), 16) % 100 < candidate_pct else "champion v1.0.0"
def pick_model(review):
    return "claude-haiku-4-5" if len(review) < 100 else "claude-sonnet-4-6"
users = ["user_{:04d}".format(i) for i in range(1000)]
cand = sum(1 for u in users if bucket(u).startswith("candidate"))
print("A/B split over {} users: {} champion, {} candidate  (~{:.0%} routed to the candidate).".format(
      len(users), len(users) - cand, cand, cand / len(users)))
same = [bucket("user_0042") for _ in range(3)]                 # DETERMINISTIC: same user, same bucket, every time
print("user_0042 asked 3x -> {}  (never flips - so A/B metrics stay clean).".format(", ".join(same)))
short, long = "Cold fries.", "The driver left it at the wrong building entirely and it took another full hour with support to sort out the mix-up."
print("model route: {!r} ({} chars) -> {}".format(short, len(short), pick_model(short)))
print("model route: long review ({} chars) -> {}".format(len(long), pick_model(long)))
print()
print("Deterministic bucketing keeps A/B clean (a user never switches versions); length routing sends cheap traffic to Haiku.")

# Output:
# A/B split over 1000 users: 899 champion, 101 candidate  (~10% routed to the candidate).
# user_0042 asked 3x -> champion v1.0.0, champion v1.0.0, champion v1.0.0  (never flips - so A/B metrics stay clean).
# model route: 'Cold fries.' (11 chars) -> claude-haiku-4-5
# model route: long review (116 chars) -> claude-sonnet-4-6
#
# Deterministic bucketing keeps A/B clean (a user never switches versions); length routing sends cheap traffic to Haiku.

**Explanation**

This is a measurement harness for the router, not a model call. It exercises `bucket` across a thousand synthetic users to show the ~90/10 split, then asks for the same user three times to demonstrate determinism. The load-bearing detail is the use of `hashlib.md5` rather than Python's built-in `hash()`, which is salted per process and would re-bucket the same user after a restart.

**How the code works - step by step**
- **`bucket`** - version routing: `md5(user_id) % 100 < candidate_pct` sends ~10% to the candidate, the rest to the champion.
- **`pick_model`** - model routing: length under 100 chars picks Haiku, otherwise Sonnet (the cost lever from 14.5).
- **`users` + `cand`** - generate 1000 user ids and count how many bucket to the candidate, showing the split.
- **`same`** - call `bucket("user_0042")` three times to prove the same user always lands in the same lane.
- **`short` / `long` prints** - run a short and a long review through `pick_model` to show the length cutoff in action.

**In one line:** hash the user for a stable version lane, measure the review length for the model.

**What you'll see:** The A/B split over 1000 users prints as roughly 899 champion / 101 candidate (~10%); `user_0042` returns `champion v1.0.0` all three times; the short review (11 chars) routes to Haiku and the long one (116 chars) to Sonnet - closing on why deterministic bucketing keeps A/B metrics clean.

## 3 - The registry: versioned prompts, cacheable prefix

Prompts are not string literals scattered through code - they are versioned files rendered into two parts: a STABLE prefix (system + few-shot) and a VARIABLE suffix (the review). This cell renders both versions and shows the prefix - the cacheable part - growing while the suffix stays the review.

In [ ]:
# THE PROMPT REGISTRY: versioned prompts loaded from YAML, rendered with the review, the stable prefix marked cacheable.
SYSTEM = "Classify a food-delivery review into exactly one of: refund, delivery, food_quality, service, other."
REGISTRY = {                                                   # loaded from prompts/quickbite-classifier-v{ver}.yaml
    "v1.0.0": {"system": SYSTEM, "few_shot": []},              # baseline, no examples
    "v1.1.0": {"system": SYSTEM, "few_shot": [                 # +3 few-shot: ambiguous, Spanish, multi-issue
        "'Money never came back' -> refund", "'Llego frio' -> food_quality", "'Late AND cold' -> delivery"]},
}
def render(version, review):
    p = REGISTRY[version]
    prefix = p["system"] + ("".join("\n" + ex for ex in p["few_shot"]))   # STABLE across calls -> cache_control here
    return {"prompt_version": version, "cacheable_prefix": prefix, "variable_suffix": "Review: " + review}
msg = render("v1.1.0", "Refund never processed.")
print("prompt_version:", msg["prompt_version"], "| prefix", len(msg["cacheable_prefix"]), "chars (cache_control=ephemeral, ~5-min TTL)")
print("cacheable prefix:", msg["cacheable_prefix"].replace("\n", "  |  "))
print("variable suffix (never cached):", repr(msg["variable_suffix"]))
print("v1.0.0 prefix", len(render("v1.0.0", "x")["cacheable_prefix"]), "chars (no few-shot) vs v1.1.0", len(msg["cacheable_prefix"]), "chars (+3 examples).")
print()
print("The system + few-shot block is identical every call, so it caches; only the review varies. The version tags every trace.")

# Output:
# prompt_version: v1.1.0 | prefix 191 chars (cache_control=ephemeral, ~5-min TTL)
# cacheable prefix: Classify a food-delivery review into exactly one of: refund, delivery, food_quality, service, other.  |  'Money never came back' -> refund  |  'Llego frio' -> food_quality  |  'Late AND cold' -> delivery
# variable suffix (never cached): 'Review: Refund never processed.'
# v1.0.0 prefix 100 chars (no few-shot) vs v1.1.0 191 chars (+3 examples).
#
# The system + few-shot block is identical every call, so it caches; only the review varies. The version tags every trace.

**Explanation**

This cell makes prompt versioning concrete. `render()` splits each prompt into a prefix that is byte-identical on every call (so it can carry `cache_control` and be discounted) and a suffix that changes per request. The candidate `v1.1.0` adds three few-shot examples deliberately chosen to target the golden set's hard cases - ambiguous, Spanish, and multi-issue reviews.

**How the code works - step by step**
- **`SYSTEM`** - the shared instruction naming the five categories (refund, delivery, food_quality, service, other).
- **`REGISTRY`** - stands in for the YAML files: `v1.0.0` has no examples; `v1.1.0` carries three few-shot examples.
- **`render`** - assembles the STABLE prefix (system + few-shot, joined by newlines) and the VARIABLE suffix (`Review: ` + text), returning both plus the version tag.
- **`msg` + prints** - render `v1.1.0`, then print the prefix length, the prefix contents, and the never-cached suffix.
- **Final length compare** - contrasts the `v1.0.0` prefix (no few-shot) with the longer `v1.1.0` prefix (+3 examples).

**In one line:** stable system+few-shot prefix (cache it) + variable review suffix (never cache it), version-tagged.

**What you'll see:** It prints `prompt_version: v1.1.0` with a 191-char cacheable prefix (ephemeral, ~5-min TTL), the full prefix contents with the three few-shot examples, the variable suffix `'Review: Refund never processed.'`, and a comparison of the 100-char `v1.0.0` prefix vs the 191-char `v1.1.0` prefix - closing on why only the prefix caches and the version tags every trace.

## 4 - Instrumentation: no trace, no LLMOps

Instrumentation is the foundation every other subsystem reads from. Every call must emit ONE trace record with a fixed field set following the OpenTelemetry GenAI conventions from 14.1. This cell builds that record for a Haiku call, checks all twelve fields are present, and computes the cost from the token counts.

In [ ]:
# INSTRUMENTATION: every call emits ONE trace record with the field set the dashboard reads (OpenTelemetry GenAI conventions, 14.1).
REQUIRED = ["trace_id", "gen_ai.request.model", "prompt_version", "user_id", "gen_ai.usage.input_tokens",
            "gen_ai.usage.output_tokens", "cost_usd", "latency_ms", "category", "confidence", "cache_read_tokens", "env"]
PRICE = {"claude-haiku-4-5": (0.80, 4.00), "claude-sonnet-4-6": (3.00, 15.00)}   # $/1M (input, output), illustrative
def build_trace(user_id, model, version, in_tok, out_tok, cache_read, category, confidence):
    pin, pout = PRICE[model]
    cost = (in_tok * pin + out_tok * pout) / 1e6
    return {"trace_id": "tr_" + user_id, "gen_ai.request.model": model, "prompt_version": version, "user_id": user_id,
            "gen_ai.usage.input_tokens": in_tok, "gen_ai.usage.output_tokens": out_tok, "cost_usd": round(cost, 6),
            "latency_ms": 420, "category": category, "confidence": confidence, "cache_read_tokens": cache_read, "env": "prod"}
tr = build_trace("user_8842", "claude-haiku-4-5", "v1.1.0", 1850, 12, 1800, "refund", 0.94)
missing = [f for f in REQUIRED if f not in tr]
print("trace field check: {}/{} required fields present, missing: {}".format(
      len([f for f in REQUIRED if f in tr]), len(REQUIRED), missing or "none - complete"))
for k in ["gen_ai.request.model", "prompt_version", "gen_ai.usage.input_tokens", "cache_read_tokens", "cost_usd", "category"]:
    print("  {:<28} {}".format(k, tr[k]))
print()
print("No trace, no LLMOps: this one record is what the dashboard, the A/B metrics, the cost report, and drift detection ALL read.")

# Output:
# trace field check: 12/12 required fields present, missing: none - complete
#   gen_ai.request.model         claude-haiku-4-5
#   prompt_version               v1.1.0
#   gen_ai.usage.input_tokens    1850
#   cache_read_tokens            1800
#   cost_usd                     0.001528
#   category                     refund
#
# No trace, no LLMOps: this one record is what the dashboard, the A/B metrics, the cost report, and drift detection ALL read.

**Explanation**

This is a validation harness for the trace schema, not a model call. It constructs the flat trace record, checks it against a list of twelve required fields, and derives `cost_usd` from the input/output token counts and the model's price. The core idea is that the dashboard, the A/B metrics, the cost report, and drift detection are ALL just readers of this one record - so a single missing field silently breaks a downstream reader.

**How the code works - step by step**
- **`REQUIRED`** - the twelve mandatory fields, following the OTel GenAI naming (`gen_ai.request.model`, `gen_ai.usage.input_tokens`, etc.).
- **`PRICE`** - illustrative per-million-token input/output prices for Haiku and Sonnet.
- **`build_trace`** - computes cost from tokens x price and returns the full flat record (ids, model, version, user, token counts, cost, latency, category, confidence, cache-read tokens, env).
- **`tr` + `missing`** - build a trace for a cached Haiku refund call, then verify no required field is absent.
- **Field loop** - print a representative subset of fields (model, version, input tokens, cache-read tokens, cost, category).

**In one line:** compute cost from tokens, emit one twelve-field record, verify nothing is missing.

**What you'll see:** It reports `12/12 required fields present, missing: none - complete`, prints six key fields (Haiku, `v1.1.0`, 1850 input tokens, 1800 cache-read tokens, cost `0.001528`, category `refund`), and closes with the line that this one record is what the dashboard, A/B metrics, cost report, and drift detection all read.

## 5 - The eval harness: golden set + judge

This is what lets you claim a prompt change is *better*, not just *different*. Run a hand-labeled golden set against each prompt version and grade every case with a multi-dimension LLM judge. This cell runs a representative 10-case slice against both versions and shows the few-shot candidate lifting exactly the hard cases.

In [ ]:
# THE EVAL HARNESS: run the golden set against a prompt version, grade each with the multi-dim judge, aggregate the pass rate.
GOLDEN = [   # a representative 10-case slice of the 50-row hand-labeled set (5 easy, 5 hard); the full set lands ~80% / ~92%
    ("Refund never arrived", "refund", "easy"), ("Where is my order?", "delivery", "easy"),
    ("Cold and soggy fries", "food_quality", "easy"), ("The driver was rude", "service", "easy"),
    ("Great food, fast delivery", "other", "easy"), ("Charged twice, still no food", "refund", "hard"),
    ("Two hours late and stone cold", "delivery", "hard"), ("Llego frio y muy tarde", "food_quality", "hard"),
    ("Driver polite but the meal was inedible", "food_quality", "hard"), ("The app keeps crashing on checkout", "other", "hard")]
# deterministic predicted category per version; v1.0.0 (no few-shot) misses 2 hard cases, v1.1.0 (few-shot) fixes 1 of them
PRED = {"v1.0.0": ["refund", "delivery", "food_quality", "service", "other", "refund", "delivery", "delivery", "service", "other"],
        "v1.1.0": ["refund", "delivery", "food_quality", "service", "other", "refund", "delivery", "food_quality", "service", "other"]}
def run_golden(version):                                        # judge grades correctness (+ format + calibration); here correctness drives it
    passed = sum(1 for (rev, exp, diff), pred in zip(GOLDEN, PRED[version]) if pred == exp)
    hard_pass = sum(1 for (rev, exp, diff), pred in zip(GOLDEN, PRED[version]) if pred == exp and diff == "hard")
    return passed, len(GOLDEN), hard_pass
for v in ["v1.0.0", "v1.1.0"]:
    p, n, hp = run_golden(v)
    print("{}: {}/{} = {:.0%} pass   (hard slice {}/5)".format(v, p, n, p / n, hp))
print()
print("The candidate (few-shot) lifts the hard, ambiguous, and multilingual cases - exactly what the few-shot examples target.")
print("The judge scores correctness + format + calibration on a 0-2 rubric with a DIFFERENT-family model; pass = all dims >= 1.")

# Output:
# v1.0.0: 8/10 = 80% pass   (hard slice 3/5)
# v1.1.0: 9/10 = 90% pass   (hard slice 4/5)
#
# The candidate (few-shot) lifts the hard, ambiguous, and multilingual cases - exactly what the few-shot examples target.
# The judge scores correctness + format + calibration on a 0-2 rubric with a DIFFERENT-family model; pass = all dims >= 1.

**Explanation**

This cell simulates the eval run with pre-computed predictions so it stays keyless - in the real project the judge (a stronger, different-family model) grades each output on correctness, format, and calibration. The golden set deliberately mixes 5 easy and 5 hard cases (ambiguous, multilingual, multi-issue); the whole point is that an all-easy set where everything passes tells you nothing.

**How the code works - step by step**
- **`GOLDEN`** - a 10-case slice of the 50-row hand-labeled set: `(review, expected, difficulty)` tuples, 5 easy and 5 hard.
- **`PRED`** - deterministic predicted categories per version; `v1.0.0` misses two hard cases (the Spanish and multi-issue ones), `v1.1.0` fixes one of them.
- **`run_golden`** - grades a version by comparing predictions to expected labels, counting total passes and hard-slice passes (correctness stands in for the full multi-dim judge here).
- **Version loop** - runs both `v1.0.0` and `v1.1.0` and prints pass rate plus hard-slice score.

**In one line:** run the same paper against each version, count passes overall and on the hard slice.

**What you'll see:** It prints `v1.0.0: 8/10 = 80% pass (hard slice 3/5)` and `v1.1.0: 9/10 = 90% pass (hard slice 4/5)`, then two lines noting the candidate lifts the hard/ambiguous/multilingual cases its few-shot examples target and that the judge scores correctness+format+calibration on a 0-2 rubric with a different-family model (pass = all dims >= 1).

## 6 - The CI gate: block the bad merge

This is the practice most teams have never reached: a CI gate that blocked its own pull request. On every PR touching `prompts/`, the golden set runs and the merge is BLOCKED unless three independent checks pass - a pass-rate floor, a max regression vs main, and a cost ceiling. This cell runs three scenarios through the gate.

In [ ]:
# THE CI EVAL GATE: on every PR touching prompts/, run the golden set and BLOCK the merge unless THREE checks pass.
FLOOR, MAX_REGRESSION, MAX_COST_INCREASE = 0.90, 0.02, 0.30    # floor 90%, max 2pp regression vs main, max +30% cost
def gate(candidate_pass, main_pass, cost_ratio):
    checks = {"floor >= 90%": candidate_pass >= FLOOR,
              "regression <= 2pp vs main": candidate_pass >= main_pass - MAX_REGRESSION,
              "cost <= +30%": cost_ratio <= 1 + MAX_COST_INCREASE}
    return all(checks.values()), checks
SCENARIOS = [("candidate v1.1.0", 0.90, 0.80, 1.00), ("a typo'd bad prompt", 0.83, 0.80, 1.00), ("a slow expensive rewrite", 0.93, 0.80, 1.55)]
for name, cand, main, cost in SCENARIOS:
    ok, checks = gate(cand, main, cost)
    reason = "" if ok else "  <- " + ", ".join(k for k, v in checks.items() if not v) + " failed"
    print("{:<24} pass {:.0%}, cost x{:.2f}  ->  {}{}".format(name, cand, cost, "MERGE" if ok else "BLOCK", reason))
print()
print("A green eval suite is the ship gate (eval-driven development). The candidate JUST clears the 90% floor; the bad PR and the")
print("expensive rewrite are each blocked, with the failed check named as a PR comment. This is the gate that makes you employable.")

# Output:
# candidate v1.1.0         pass 90%, cost x1.00  ->  MERGE
# a typo'd bad prompt      pass 83%, cost x1.00  ->  BLOCK  <- floor >= 90% failed
# a slow expensive rewrite pass 93%, cost x1.55  ->  BLOCK  <- cost <= +30% failed
#
# A green eval suite is the ship gate (eval-driven development). The candidate JUST clears the 90% floor; the bad PR and the
# expensive rewrite are each blocked, with the failed check named as a PR comment. This is the gate that makes you employable.

**Explanation**

This is the decision logic that turns the eval from a manual script into an automatic wall. `gate()` applies three checks and merges only if all pass - the key case being a quality win that blows the budget, which is blocked on cost alone. The three scenarios are chosen to fire each outcome: a clean merge, a floor failure, and a cost failure.

**How the code works - step by step**
- **`FLOOR, MAX_REGRESSION, MAX_COST_INCREASE`** - the three thresholds: 90% floor, 2pp max regression vs main, +30% max cost.
- **`gate`** - evaluates the three checks into a dict and returns `all(checks.values())` plus the per-check breakdown.
- **`SCENARIOS`** - three `(name, candidate_pass, main_pass, cost_ratio)` cases: the real candidate, a typo'd bad prompt, and a slow expensive rewrite.
- **Scenario loop** - runs each through the gate, names the failed check(s) when blocked, and prints MERGE or BLOCK.

**In one line:** three thresholds, all must pass, fail any one and the merge is blocked with the reason named.

**What you'll see:** The candidate v1.1.0 (90% pass, cost x1.00) prints `MERGE`; the typo'd prompt (83%) prints `BLOCK <- floor >= 90% failed`; the expensive rewrite (93% pass but cost x1.55) prints `BLOCK <- cost <= +30% failed` - showing a quality win cannot smuggle in a cost blow-up, with the failed check posted as a PR comment.

## 7 - Ship it: project the cost, score the checklist

The last step turns a working pipeline into a shippable, defensible artifact. Project the monthly cost at 1x / 10x / 100x traffic (routing and caching already applied to the per-request cost), then score the go-live checklist a senior reviewer will run in five minutes. This cell does both.

In [ ]:
# SHIP IT: project the monthly cost at 1x / 10x / 100x traffic (routing + caching applied) and score the go-live checklist.
reqs_per_day = 5000
COST_PER_REQ = 0.00042        # $ blended per request AFTER length routing (cheap traffic -> Haiku) + prompt caching (from the traces)
def monthly(mult): return reqs_per_day * 30 * COST_PER_REQ * mult
print("Cost projection (blended, after routing + caching):")
for mult, label in [(1, "today"), (10, "10x"), (100, "100x")]:
    print("  {:<6} {:>12,} req/mo  ->  ${:>9.2f}/mo".format(label, reqs_per_day * 30 * mult, monthly(mult)))
CHECKLIST = [("endpoint returns the JSON contract", True), ("every call traced (12 fields)", True),
             ("2 prompt versions, deterministic A/B", True), ("50-row golden set + multi-dim judge", True),
             ("CI gate blocks a real regression", True), ("routing + caching on, verified in traces", True),
             ("dashboard deployed + README cost analysis", True), ("no committed secrets; golden set + traces PII-scrubbed", True)]
done = sum(1 for _, ok in CHECKLIST if ok)
print()
print("Go-live checklist: {}/{} green.".format(done, len(CHECKLIST)))
print("A senior reviewer reads this repo in five minutes and sees production discipline most teams never reach. Ship it.")

# Output:
# Cost projection (blended, after routing + caching):
#   today       150,000 req/mo  ->  $    63.00/mo
#   10x       1,500,000 req/mo  ->  $   630.00/mo
#   100x     15,000,000 req/mo  ->  $  6300.00/mo
#
# Go-live checklist: 8/8 green.
# A senior reviewer reads this repo in five minutes and sees production discipline most teams never reach. Ship it.

**Explanation**

This is a projection-and-scorecard cell, not a model call. It scales a blended per-request cost - already discounted by length routing and prompt caching from the traces - out to monthly burn at three traffic multiples, then walks an eight-item go-live checklist. The point is that a stranger can open the repo cold and see production discipline: contract, trace, A/B, eval, gate, cost levers, dashboard, no secrets.

**How the code works - step by step**
- **`reqs_per_day` / `COST_PER_REQ`** - the traffic assumption (5000/day) and the blended per-request cost after routing + caching.
- **`monthly`** - scales daily requests to a monthly dollar figure at a given traffic multiple.
- **Cost loop** - prints monthly request volume and cost at 1x (today), 10x, and 100x.
- **`CHECKLIST`** - the eight go-live items (contract, 12-field trace, deterministic A/B, 50-row golden set + judge, CI gate that blocks a real regression, routing+caching verified, deployed dashboard + README cost analysis, no secrets + PII-scrubbed golden set), all marked done.
- **`done` + prints** - count green items and print the ship verdict.

**In one line:** scale the per-request cost to monthly burn, then confirm all eight go-live items are green.

**What you'll see:** It prints the cost projection - $63.00/mo today (150,000 req), $630.00/mo at 10x, $6300.00/mo at 100x - then `Go-live checklist: 8/8 green.` and the closing line that a senior reviewer reads this repo in five minutes and sees production discipline most teams never reach.

Across seven runnable cells you wired an entire production feature from parts you built earlier in Module 14: the router makes two deterministic decisions, the registry serves versioned prompts with a cacheable prefix, every call emits one twelve-field trace, the eval harness turns that into a better-or-worse signal, and the CI gate turns the signal into a merge decision - then the cost projection and checklist make it shippable. The through-line is two wired paths: can you see what every request actually did, and does every prompt change have to earn its merge? Take the same skeleton to the separate Project Course (a full agentic-RAG build) and to Module 15, the ethics and responsible-AI layer that sits on top of everything you just shipped.